## Step5-Final-Adjustments-so

Makes final adjustments to the salinity files as follows:
1. Making time axis CF compliant and consistent wiht other variables
2. Splitting into annual files
3. Fixing attributes
4. Setting final file naming convention
5. Adding a 2300 file that is the same as 2299, in the case that 2300 is missing from the CMIP output (as is the case for CESM2-WACCM)
6. Appending first 11 months of 1850 file with that of 1851

#### Required files/inputs
- CMIP bias-corrected monthly salinity files, e.g. *so_QDMCorrected-ISMIP_Grid-CESM2-WACCM-1850_2014-20260410.nc*, (produced in step 4, last 8 digits = date the file was generated)

#### Outputs
- The final output: CMIP monthly salinity files, split into annual files e.g. *so_GrIS_CESM2-WACCM_historical_ocean-1000m_v2_1850.nc*

In [ ]:
# ---- Imports ----

import os
import xarray as xr
import numpy as np
import cftime


In [ ]:
# ---- Inputs ----

SelModel = 'CESM2-WACCM'
chosen_scenario = 'ssp585'
version = 'v2'

file_path = f"/vscratch/grp-sophien/naveen-temp/gris-iceocean-outfiles/so/{chosen_scenario}/"

files = [
         file_path + "so_QDMCorrected-ISMIP_Grid-CESM2-WACCM-1850_2014-20260410.nc",
         file_path + "so_QDMCorrected-ISMIP_Grid-CESM2-WACCM-2015_2100-20260410.nc",
         file_path + "so_QDMCorrected-ISMIP_Grid-CESM2-WACCM-2201_2299-20260410.nc",
         file_path + "so_QDMCorrected-ISMIP_Grid-CESM2-WACCM-2201_2299-20260410.nc"
      ]


In [ ]:
# ---- Defining function for processing ----

def final_so_adjustments(sofile):

    ds = xr.open_dataset(sofile)

    # change time to mid month
    days_in_month = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
    mid_month = np.ceil(days_in_month / 2).astype(int)

    dates = []
    for date in ds['time']:
        year = int(date.dt.year)
        month = int(date.dt.month)
        dates.append(cftime.DatetimeNoLeap(year, month, mid_month[month-1]))

    ds = ds.assign_coords(time=dates)

    # attributes
    ds["x"].attrs.update({
        "long_name": "Cartesian x-coordinate",
        "units": "meter",
        "axis": "X"
    })
    ds["y"].attrs.update({
        "long_name": "Cartesian y-coordinate",
        "units": "meter",
        "axis": "Y"
    })

    ds["crs"] = xr.DataArray(np.int32(0))
    ds["crs"].attrs = {
            "grid_mapping_name": "polar_stereographic",
            "straight_vertical_longitude_from_pole": -45.0,
            "latitude_of_projection_origin": 90.0,
            "standard_parallel": 70.0,
            "false_easting": 0.0,
            "false_northing": 0.0,
            "semi_major_axis": 6378137.0,
            "inverse_flattening": 298.257223563
    }

    # link CRS to variable
    ds["so"].attrs["grid_mapping"] = "crs"

    ds["so"].attrs.update({
        "long_name": "sea water salinity",
        "units": "psu"
    })

    startyear = ds['time'].values[0].year
    endyear = ds['time'].values[-1].year

    for year in range(startyear, endyear + 1):
        if year <= 2014:
            scenario = 'historical'
        else:
            scenario = chosen_scenario

        ds_year = ds.sel(
            time=slice(
                cftime.DatetimeNoLeap(year, 1, 1),
                cftime.DatetimeNoLeap(year, 12, 31)
            )
        )

        outfile = f'/vscratch/grp-sophien/naveen-temp/GrIS/{SelModel}/{scenario}/ocean-1000m/so/{version}/' # path maybe differnt
        filename = f"so_GrIS_{SelModel}_{scenario}_ocean-1000m_{version}_{year}.nc"

        if os.path.exists(outfile + filename):
          os.remove(outfile + filename)

        os.makedirs(outfile, exist_ok=True)

        comp = dict(zlib=True, complevel=9)
        encoding = {var: comp for var in ds.data_vars}

        ds_year.to_netcdf(outfile + filename, format="NETCDF4", encoding=encoding)

In [ ]:
# ---- Processing files ----

for sofile in files:
    print(f'Processing {sofile}')
    final_so_adjustments(sofile)
    print('')

In [ ]:
# ---- Fix 2300 ----
if chosen_scenario != 'ssp370':
    so2299 = f"/vscratch/grp-sophien/naveen-temp/GrIS/{SelModel}/{chosen_scenario}/ocean-1000m/so/{version}/so_GrIS_{SelModel}_{chosen_scenario}_ocean-1000m_{version}_2299.nc"
    so2300 = f"/vscratch/grp-sophien/naveen-temp/GrIS/{SelModel}/{chosen_scenario}/ocean-1000m/so/{version}/so_GrIS_{SelModel}_{chosen_scenario}_ocean-1000m_{version}_2300.nc"

    ds_2299 = xr.open_dataset(so2299)
    ds_2300 = ds_2299.copy()

    new_times = [
    cftime.DatetimeNoLeap(2300, int(t.dt.month), int(t.dt.day))
    for t in ds_2299['time']
    ]

    ds_2300 = ds_2300.assign_coords(time=new_times)
    ds_2300.attrs['Comment'] = "Copy of year 2299 - with updated timestamp"

    comp = dict(zlib=True, complevel=9)
    encoding = {var: comp for var in ds_2300.data_vars}

    ds_2300.to_netcdf(so2300, format="NETCDF4", encoding=encoding)

In [ ]:
# ---- Fix 1850 ----
# historical files comes from scenario ssp370

if chosen_scenario == 'ssp370':
    so1851 = f"/vscratch/grp-sophien/naveen-temp/GrIS/{SelModel}/historical/ocean-1000m/so/{version}/so_GrIS_{SelModel}_historical_ocean-1000m_{version}_1851.nc"
    so1850 = f"/vscratch/grp-sophien/naveen-temp/GrIS/{SelModel}/historical/ocean-1000m/so/{version}/so_GrIS_{SelModel}_historical_ocean-1000m_{version}_1850.nc"

    tmp1850 = so1850.replace(".nc", "_tmp.nc")


    ds1851 = xr.open_dataset(so1851)
    ds1850 = xr.open_dataset(so1850)

    jan_nov_1851 = ds1851.sel(time=ds1851.time.dt.month <= 11)
    dec_1850 = ds1850.sel(time=ds1850.time.dt.month == 12)


    ds_out = xr.concat([jan_nov_1851,dec_1850], dim="time")

    ds_out = ds_out.assign_coords(
        time=[
            cftime.DatetimeNoLeap(1850, t.month, t.day)
            for t in ds_out.time.values
        ]
    )

    ds_out.attrs["Comment"] = " copy of Jan–Nov 1851 in 1850"

    comp = dict(zlib=True, complevel=9)
    encoding = {var: comp for var in ds_out.data_vars}

    ds_out.to_netcdf(tmp1850, format="NETCDF4", encoding=encoding)

    os.replace(tmp1850, so1850)